> Translation note: Markdown teaching text has been translated to English. Code cells are preserved to avoid breaking execution; previous outputs were cleared.\n

# Implementing a GPT-Style Language Model from Scratch

## Project Overview: A Journey Toward Intelligent Text

Imagine that we have a large pile of text in front of us, initially just a chaotic mass of symbols. If we want to teach a machine to understand it, generate it, or even “think” with it, what should the first step be?

**Core Question 1: How can a computer “understand” text?**

In the digital world, text is only a sequence of characters. How can we transform these abstract symbols into numerical representations that a machine can process and that can carry meaning? This leads to our starting point: **Data Representation**, especially how to build a “vocabulary” so that every character or text chunk has its own identity.

**Core Question 2: How can we make a machine “predict” the next word?**

Once text has numerical identities, how can we build the simplest possible model that guesses the next likely word from the information already available? We will begin with a seemingly naive but fundamentally important **Baseline Model** and reveal the first principles behind machine learning for language.

**Core Question 3: How can we go beyond “short-sighted” prediction?**

Predicting the next word from only the most recent information is far from enough. The subtlety of language lies in context. We need a mechanism that allows the machine, while making a prediction, to look back and “attend” to all relevant previous information. This is the central idea of **Self-Attention**, which allows the model to flexibly weight the importance of every previous word when processing each word.

**Core Question 4: How can we build a language brain that can “think”?**

Self-attention is like a “connection” between neurons, but a single connection is not enough to form intelligence. We need to organize these connections into powerful units of “thinking.” This involves building a **Transformer Block**, which combines self-attention, feed-forward networks, residual connections, and layer normalization to process information across multiple dimensions. Finally, we will stack these “thinking units” to build a complete **GPT-style language model**, giving it deeper language understanding and generation ability.

**Core Question 5: How can a machine adapt to different tasks?**

A truly intelligent language model should not be limited to generating text. It should also solve practical problems such as sentiment analysis and dialogue. We will also discuss how to transfer the power of a pretrained model **to downstream tasks** through fine-tuning and related methods, making it more practical and versatile.

Through this exploration, we will gradually uncover the mechanisms behind Transformer models. Starting from the most basic building blocks, we will build an intelligent system that can understand and generate complex language.

---


## Phase 1: Data Representation

Before building any complex architecture such as a Transformer, we must solve a basic question: **How can a computer recognize text?**

Imagine that you only have a text string. To the computer, it is just a binary sequence. To make it usable by a model, we need to build a Vocabulary that maps each unique character to an integer ID. This is the simplest form of **Tokenization**.


In [ ]:
from pathlib import Path
import urllib.request

file_path = Path("input.txt")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not file_path.exists():
    print("文件不存在，开始下载...")
    urllib.request.urlretrieve(url, file_path)
    print("下载完成！")
else:
    print("文件已存在，跳过下载。")

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"数据集总字符数: {len(text)}")

# 获取所有唯一字符并建立词表
chars = sorted(list(set(text)))  # 为什么sorted(list(set(text)))在这里能实现获取所有字符的功能。
vocab_size = len(chars)
print(f"词表大小: {vocab_size}")
print(f"所有字符: {''.join(chars)}")

Q: Why does `sorted(list(set(text)))` obtain all characters here?

A: A `set` is an unordered data structure whose elements are unique. When you put the entire text into a set, it automatically removes all repeated characters and keeps only the unique characters that appear. This is how we obtain the “vocabulary.” The resulting vocabulary consists of English letters plus punctuation marks.

In Python, the default sorting behavior of `sorted()` for strings or characters is ascending order based on Unicode code points. In deep learning, consistency matters more than almost anything else. As long as we use `sorted()`, no matter which machine we run on, `'a'` will always map to the same numerical ID.


### Exercise 1.1

Now we have the character list `chars`. To implement `encode` from text to numbers and `decode` from numbers back to text, we need to build two mapping tables.

**TODO**: Complete the code below, build the mapping dictionaries, and think about this question: why do we choose **character-level** tokenization here instead of word-level tokenization? What effect does this have on the vocabulary size?


In [ ]:
# TODO: 构建 string -> integer (stoi) 和 integer -> string (itos) 的映射

def encode(s):
    # TODO: 将字符串 s 转换为数字列表
    l = []
    for char in s:
        l.append(chars.index(char))
    return l

def decode(l):
    # TODO: 将数字列表 l 转换回字符串
    s = []
    for i in l:
        c = chars[i]
        s.append(c)
    s = ''.join(s)
    return s

# 参考答案
# def encode(s):
#     stoi = {c: i for i, c in enumerate(chars)}
#     return [stoi[c] for c in s]

# def decode(l):
#     itos = {i: c for i, c in enumerate(chars)}
#     return ''.join([itos[i] for i in l])

# 测试代码（请在补全后取消注释并运行）
test_str = "hello world"
encoded = encode(test_str)
print(f"Encoded: {encoded}")
print(f"Decoded: {decode(encoded)}")

### Exercise 1.2: Turning Data into Tensors

In PyTorch, all data must ultimately exist as tensors. We will now convert the entire Shakespeare dataset into a large integer sequence.


In [ ]:
import torch

# TODO: 将整个数据集 encode 并转换为 torch.Tensor
# 别忘了用 torch.tensor() 包装 encode 后的结果
data = torch.tensor(encode(text), dtype=torch.long)

# 打印前 100 个字符的 Tensor 形式
print(data[:100])
print(f"Tensor 形状: {data.shape}")

**Guided reflection**:
After converting the data into a Tensor, we no longer need the original string. In this Tensor, what physical meaning does each number represent? Why do we choose `torch.long` instead of `torch.float`?


---

## Phase 2: Baseline Model

### Exercise 2.1: Train/Val Split

To evaluate model quality, we need to split the dataset into a **training set** and a **validation set**.


In [ ]:
# TODO: 计算切分点索引 (90% 训练, 10% 验证)
n = int(0.9 * len(data))
train_data = data[0:n-1]
val_data = data[n:]

print(f"训练集长度: {len(train_data)}")
print(f"验证集长度: {len(val_data)}")

### Exercise 2.2: Data Batching

We need to randomly sample from the dataset. For an input sequence `x`, its target `y` is the same sequence shifted one position forward.


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # 每次并行处理多少个序列
block_size = 8 # 每个序列的最大长度

def get_batch(split, batch_size):
    # split 可以是 'train' 或 'val'
    data_subset = train_data if split == 'train' else val_data

    # TODO: 随机生成 batch_size 个起始索引
    ix = torch.randint(0, len(data_subset) - block_size, (batch_size,))

    # TODO: 根据 ix 提取 x 和 y
    x = [data_subset[i:i+block_size] for i in ix]
    y = [data_subset[i+1:i+block_size+1] for i in ix]
    return torch.stack(x), torch.stack(y)

xb, yb = get_batch('train', batch_size)
print('inputs (xb):')
print(xb.shape, xb)
print('targets (yb):')
print(yb.shape, yb)

**Guided reflection**:
Look at the first row of `xb` and the first row of `yb`. If `xb[0]` is `[18, 47, 56]`, what should the corresponding `yb[0]` be? How does this shifted design express the task objective of “predicting the next character”?

A: `yb[0]=[47, 56, xx (the next character)]`. When the input is `x = 18`, we want the model to output the next character, so `y = 47`.


**Guided reflection**:
When we later train the Transformer, does the model read all one million characters at once, or does it learn in many small meals? If the split point is not exactly at the end of a sentence, will it matter?

A: The model learns in small batches. It is not a problem if the split point is not exactly at the end of a sentence. As long as it sees enough data, it can learn broader contextual relationships.


### Exercise 2.3: Bigram Language Model

We will use PyTorch to build the most basic model. In this model, Tokens do not “communicate” with each other. Each Token only uses its own index to look up an Embedding vector, and this vector directly represents its prediction scores for the next character. Note that the token index itself does not contain a distance relationship; only the embedding vector may contain such relationships.

The model gives a prediction for the next character, but we still need to judge: **Is the prediction accurate?**
For example, suppose the true next character is `e`. If the model assigns probability `0.9` to `e`, the prediction is good. If it assigns only `0.1`, the model has produced a prediction but has not assigned high probability to the correct answer.

Therefore, we need a numerical value to measure the gap between the model prediction and the true answer. This value is called **loss**.
The larger the loss, the worse the model prediction. The smaller the loss, the closer the prediction is to the true answer. Training a model is essentially the process of repeatedly adjusting parameters so that the loss gradually decreases.
In language modeling, the most commonly used loss function is **Cross Entropy Loss**. Its core idea is:

**The higher the probability the model assigns to the correct answer, the smaller the loss; the lower the probability, the larger the loss.**

In PyTorch, we usually use:

```python
F.cross_entropy(logits, targets)
```

Here, `logits` are the raw scores output by the model, and `targets` are the indices of the true next tokens. Note that `F.cross_entropy` automatically performs the softmax-related computation internally, so we do not need to manually convert logits into probabilities first.
For the Bigram Language Model, the model looks up a table based on the current token and obtains prediction scores for the next token. We then compare this prediction with the true next token using cross-entropy loss, and update the table through backpropagation so that the model gradually learns which characters are more likely to follow which characters.


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from torch.distributions import Categorical

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 每个 token 直接读取下一个 token 的 logits
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx 和 targets 都是 (B, T) 的张量
        # idx 是输入字符串的索引
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx 是 (B, T) 的当前索引数组
        for _ in range(max_new_tokens):
            # TODO: 1. 获取预测结果 (logits)
            # TODO: 2. 只关注最后一个时间步的结果 (Focus on the last time step)
            # TODO: 3. 使用 Softmax 转化为概率
            # TODO: 4. 从概率分布中采样得到下一个 Token
            # TODO: 5. 将采样结果拼接到 idx 并继续循环

            # logits = self.token_embedding_table(idx)[:, -1, :]
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            m = Categorical(probs)
            idx_next = m.sample()
            idx = torch.cat((idx, idx_next.unsqueeze(1)), dim=1)

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"初始 Loss: {loss}")

In [ ]:
import torch

# 创建一个 (1, 1) 的张量，内容为 0（通常是换行符），作为生成的起点
idx = torch.zeros((1, 1), dtype=torch.long)

# 调用 generate 函数生成 100 个 token
# 注意：m.generate 返回的是数字索引，我们需要用 decode 转换回文字
generated_indices = m.generate(idx, max_new_tokens=100)[0].tolist()

print("--- 未经训练的模型生成的文本 ---")
print(decode(generated_indices))
print("------------------------------")

Q: Does `nn.Embedding` only embed the input, without reaching the prediction stage yet?

A: Usually, `nn.Embedding` only turns numbers into vectors. But in our Bigram model, we use a small mathematical “trick.”

We set the Embedding dimension directly to `vocab_size`. This means that when you input the character `'A'` with ID 13, the lookup returns an array of length 65. We directly treat every number in this array as the “possibility” or logit that the next character is the character at the corresponding position in the vocabulary.

Q: Why is the Embedding layer represented as a matrix of shape `(num_embeddings, embedding_dim)`?

A: This matrix structure is essentially a **learnable lookup table**. Each row represents the feature vector of a Token. The matrix form lets us use efficient GPU matrix operations while allowing the model to optimize these weights through gradient descent. In the Bigram model, this row vector directly serves as the prediction scores for the next character.

Q: Why not use `nn.Linear` as the embedding layer?

A: Mathematically, they are equivalent: Embedding is the same as multiplying a one-hot input by a Linear layer. But in implementation, Embedding replaces matrix multiplication with index lookup, avoiding high-dimensional sparse matrices. This greatly saves memory and improves speed.


**Guided reflection**:
When computing Loss, why do we need to apply `view` or reshaping to `logits` and `targets`? What special dimensional requirement does PyTorch’s `cross_entropy` impose on its input?

A: `F.cross_entropy` expects the input logits to be two-dimensional, with shape `(Total_Samples, Classes)`, while our model output is three-dimensional, `(Batch, Time, Classes)`. By using `view(B*T, C)`, we treat the prediction at each time step as an independent sample, satisfying the API requirements.


### Exercise 2.4: Training the Model

Garbled-looking output is normal. The parameters inside our `token_embedding_table` are currently all random numbers. To make it learn the style of Shakespeare, we need to:
1. Create an Optimizer.
2. Repeatedly obtain Batch data.
3. Compute Loss and update weights through backpropagation.

**TODO**: Use the `torch.optim.AdamW` optimizer and try to write a simple training loop, for example 1000 steps. Observe whether the Loss decreases.


In [ ]:
import torch

# 创建优化器
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3, betas=(0.9, 0.9))

batch_size = 32
for steps in range(10000): # 10000 次训练迭代

    # 1. 获取随机 Batch 数据
    xb, yb = get_batch('train', batch_size)

    # 2. 前向传播计算 Loss
    logits, loss = m(xb, yb)

    # 3. 反向传播更新参数
    optimizer.zero_grad(set_to_none=True) # 清空梯度，推荐使用 set_to_none
    loss.backward()
    optimizer.step()

    if steps % 1000 == 0:
        print(f"Step {steps}: Loss {loss.item():.4f}")

print(f"最终 Loss: {loss.item():.4f}")

In [ ]:
# 训练后再次生成，看看效果
print("--- 训练后的模型生成的文本 ---")
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

Q: Why can the Loss only drop to around 2.5?

A: There is a mathematical limitation. The Bigram model is essentially learning a conditional probability table $P(next|current)$. In the Shakespeare dataset, given that the current character is `'e'`, the next character could be `' '`, `'a'`, `'n'`, `'r'`, `'s'`, and so on. Because the model only sees this one current character, it cannot distinguish whether this `'e'` belongs to `'the'`, `'where'`, or `'hell'`. This inherent uncertainty creates a lower bound for its Loss, namely the theoretical minimum entropy.

There is also missing information. Imagine that I only show you the letter `'h'` and ask you to guess the next character. It would be hard to guess accurately. But if I show you `'t'`, `'h'`, you would be much more confident that the next character is `'e'`. The Bigram model has too narrow a “field of view” (Context length = 1), so it can never eliminate the part of the Loss caused by insufficient context.

For comparison, the current Loss around 2.5 is exactly the limit of single-character prediction. Once we move to Self-Attention, the model can see the past 8 characters or even thousands of characters at once. Then the Loss can easily fall below 2.0, and the generated text will begin to look fluent.


---

## Phase 3: Self-Attention

### 3.1 The Core Tension: The Lonely Character

In the Phase 2 Bigram model, each character only looks at itself. For example, when predicting `the`, if the model sees `h`, it does not know whether the previous character was `t` or `s`. To generate fluent text, we need a mechanism that allows the current character to **absorb** information from previous characters.

### 3.2 A Basic Method for Information Fusion: Weighted Average

The simplest approach is to make the output at position $t$ equal to the **average** of all input vectors from $0$ to $t$. In this way, position $t$ has “seen” all previous information.

However, doing this with a `for` loop in PyTorch is very slow. We will use a **small mathematical trick**: lower-triangular matrix multiplication.


### 3.3 Why Not Use a For Loop? Performance Verification

Although the matrix multiplication below may look a bit indirect, it greatly leverages the parallel power of GPUs. We compare Method 1, explicit loops, and Method 2, matrix multiplication, to see the efficiency difference.


In [ ]:
import torch
import time

# --- 核心对比：向量化 (Vectorization) 的威力 ---
B, T, C = 4, 8, 2

# 为了对比，我们需要一些随机输入数据 x
torch.manual_seed(42)
x = torch.randn(B, T, C)

# 1. 方案 1: 原始 For 循环
start = time.time()
for _ in range(100):
    xbow = torch.zeros((B, T, C))
    for b in range(B):
        for t in range(T):
            xprev = x[b, :t+1]
            xbow[b, t] = torch.mean(xprev, 0)
t1 = (time.time() - start) / 100

# 2. 方案 2: 矩阵乘法 (使用预计算的下三角权重)
# 这种方式利用了 PyTorch 的底层 C++/CUDA 优化
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)

start = time.time()
for _ in range(100):
    xbow2 = weights @ x
t2 = (time.time() - start) / 100

# --- 输出打印 ---
print(f"[性能对比]")
print(f"方案 1 (For 循环): {t1:.6f}s")
print(f"方案 2 (矩阵乘法): {t2:.6f}s (快 {t1/t2:.1f} 倍)")

print(f"\n[一致性检查]")
# 再次强调：使用 atol=1e-7 来容忍微小的浮点数舍入误差
print(f"方案 1 与 2 结果是否一致: {torch.allclose(xbow, xbow2, atol=1e-7)}")

You will find that matrix multiplication is usually more than an order of magnitude faster than a `for` loop. This is why, when implementing neural networks, especially Transformers, we try our best to convert all logic into matrix operations.


In PyTorch and NumPy, the `@` symbol means Matrix Multiplication. It is equivalent to calling `numpy.matmul()`.

In the code `weights @ x`, a weight matrix of shape `(T, T)` is multiplied in batch with data of shape `(B, T, C)`. PyTorch automatically handles the Batch dimension, and the final result has shape `(B, T, C)`. This is effectively a weighted sum that lets each time step “see” the historical information before it.


### 3.4 Information Fusion: From Mathematical Formula to Code

In self-attention, we want the character at position $t$ to “fuse” all information from positions $1$ to $t$. The simplest fusion method is to compute the **average of all past time steps**.

**1. Mathematical expression**:
If the input sequence is $x_1, x_2, \dots, x_T$, the output $a_t$ is computed as:
$$a_t = \frac{1}{t} \sum_{i=1}^{t} x_i$$
This means each time step “absorbs” its history, with uniformly distributed weights.

**2. Matrix implementation**:
In PyTorch, we implement this using a **normalized lower-triangular matrix** $W$:
$$A = W \cdot X$$

Let us use $T=3$ as an example and see what happens in $W \cdot X$:

$$W = \begin{pmatrix} 1 & 0 & 0 \\ 0.5 & 0.5 & 0 \\ 0.33 & 0.33 & 0.33 \end{pmatrix}, \quad X = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \end{pmatrix}$$

When we compute $A = W \cdot X$:
- **First row**: $a_1 = (1 \cdot x_1 + 0 \cdot x_2 + 0 \cdot x_3) = x_1$
- **Second row**: $a_2 = (0.5 \cdot x_1 + 0.5 \cdot x_2 + 0 \cdot x_3) = \frac{x_1 + x_2}{2}$
- **Third row**: $a_3 = (0.33 \cdot x_1 + 0.33 \cdot x_2 + 0.33 \cdot x_3) = \frac{x_1 + x_2 + x_3}{3}$

As you can see, each row of matrix $W$ is actually a combination of a **Mask** and a **Weight**:
1. **Lower-triangular structure**: ensures that row $t$ can only see data whose index is $\le t$, meaning it does not “look into the future.”
2. **Row normalization**: ensures that the sum of all visible historical weights is 1, meaning it computes an average.


In [ ]:
import torch

# 假设 Batch=1, Time=3, Channels=2
B, T, C = 1, 3, 2
x = torch.tensor([[[10, 20], [30, 40], [50, 60]]], dtype=torch.float)

# 1. 创建下三角全1矩阵
tril = torch.tril(torch.ones(T, T))

# 2. 归一化：让每一行的和为 1，这样矩阵乘法就变成了“求平均”
weights = tril / tril.sum(1, keepdim=True)

# 3. 矩阵乘法融合信息
# xbow 的第 t 行 = x 中前 t 行的平均值
xbow = weights @ x

print("输入 x (3个时刻):\n", x)
print("\n权重矩阵 (下三角平均):\n", weights)
print("\n融合结果 xbow (各时刻融合了历史信息):\n", xbow)

### 3.5 The Final Method: Using Softmax to Unify Formula and Code

In deep learning, we rarely perform normalization directly by division. Instead, we use `Softmax`. This method is not only mathematically cleaner but also prepares us for the “non-uniform weights” we will encounter later.

**The three steps of the Softmax approach:**
1. **Prepare**: create an all-zero matrix, meaning all historical positions are initially equally important.
2. **Masking**: set the “future” positions where `tril == 0` to $-\infty$.
3. **Activation (Softmax)**: apply `Softmax` row-wise. Since $e^0=1$ and $e^{-\infty}=0$, each row automatically becomes $[1/t, 1/t, \dots, 0]$.

This is **exactly equivalent** to the formula $a_t = \frac{1}{t} \sum_{i=1}^{t} x_i$.


In [ ]:
import torch.nn.functional as F

# 1. 初始化全 0 权重
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T, T))

# 2. 遮蔽未来：将右上角变为负无穷
weights = weights.masked_fill(tril == 0, float('-inf'))  # 将矩阵中0元素的地方替换为负无穷

# 3. 通过 Softmax 归一化
# 此时每一行的和为 1，且非零部分的值完全相等（即 1/t）
weights = F.softmax(weights, dim=-1)

# 4. 执行融合
xbow3 = weights @ x

print("Softmax 产生的权重矩阵:\n", weights)
print("\n方案 3 (Softmax) 与方案 2 (直接归一化) 是否一致:")
print(torch.allclose(xbow, xbow3, atol=1e-7))

**Deep understanding**: Why initialize with 0 and $-\infty$?

These two steps are the standard way to implement a Causal Mask:

1. **All-zero initialization**: before attention is assigned, we assume every word in the history has the **same** influence on the current time step.
2. **The useful role of $-\infty$**:
   * Mathematically, $\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$.
   * For “future time steps” that we do not want the model to see, we set $x_i$ to $-\infty$.
   * Since $e^{-\infty} = 0$, those future time steps contribute 0 to the sum, and their final probability is also exactly 0.

**Summary**: this step ensures that the model cannot “peek” at the answer during training. When predicting the $t$-th word, it must not see the $(t+1)$-th word.


### 3.6 **Advanced**: From “Simple Average” to “Single-Head Self-Attention”

In Section 3.4, we implemented a **historical average** through matrix multiplication. Now we will upgrade it so that the model can decide for itself which historical positions to “attend” to.

#### 1. Core evolution: Query, Key, and Value
Instead of rigid average weights, we introduce three linear transformations:
- **Query (Q)**: “What am I looking for?” The intention of the current time step.
- **Key (K)**: “What information do I contain?” The index-like signal of each historical time step.
- **Value (V)**: “What will I contribute if selected?” The actual feature content to be transmitted.

#### 2. Attention computation process
1. **Compute weights (Affinity)**: compute the dot product between $Q$ and $K$. If the Key of a historical time step matches the current Query well, it receives a higher weight.
2. **Normalize**: use Softmax so that the weights sum to 1.
3. **Extract content**: use the weights to compute a weighted sum over **Value (V)**, rather than averaging the original input $X$.

#### 3. Final formula
The final formula of attention in Transformers is:
$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

This is why the final step in the code is `out = wei @ v`: it is essentially extracting the useful information selected by “data-driven weights.”


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 假设输入 x 的维度
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# 1. 定义线性层
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# 2. 计算 Q, K, V
k = key(x)   # (B, T, head_size)
q = query(x) # (B, T, head_size)
v = value(x) # (B, T, head_size)

# 3. 计算注意力权重 (Affinity)
# TODO: 参照注意力公式计算权重
wei = q @ k.transpose(-2, -1)  # 交换张量K最后两个维度的位置：(batch_size, num_heads, seq_len_k, head_dim)

# TODO: 请对 wei 进行缩放 (Scaled Dot-Product Attention)
# 提示：将 wei 除以 head_size 的平方根。如果不缩放，Softmax 后的梯度会变得非常小。
wei = wei * (head_size ** -0.5)

# 4. 遮蔽未来信息与归一化
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

# 5. 最后与 Value 融合
out = wei @ v

print("输出形状:", out.shape)
print("\n最后时刻的权重分布:\n", wei[-1, -1])

Observe that the current `wei` matrix is no longer the rigid $1/t$. It is a set of numbers computed from $Q$ and $K$, which will later be optimized through training.

**Guided reflection**:
If two Tokens are highly similar, meaning their $Q$ and $K$ dot product is large, what will happen to the attention weight between them? How does this solve the “narrow field of view” problem of the Bigram model?


### 3.7 Advanced: Multi-Head Attention

Why use multiple heads? If there were only one head, the model could learn only one type of relationship at a time, such as focusing only on the relationship between verbs and nouns. But language is complex. We need the model to attend to multiple relationships simultaneously, such as syntactic relations, long-distance dependencies, and modifier relations. This is the meaning of multi-head self-attention: it allows the model to learn in parallel across different subspaces.

You can imagine multi-head attention as a group of independent “heads” running in parallel. Each head computes independently, and finally we concatenate their outputs.


In [ ]:
class Head(nn.Module):
    """ 一个单头自注意力层 """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # 计算注意力分数 (Affinity)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # 融合 Value
        v = self.value(x)
        out = wei @ v # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ 多个并行运行的自注意力头 """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # TODO: 构建多头注意力模型
        # 提示: 使用 nn.ModuleList 存放多个独立的 Head
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # 最后通过一个线性层投影回原始维度 (可选，但推荐)
        self.proj = nn.Linear(num_heads * head_size, C)

    def forward(self, x):
        # TODO: 拼接所有头的输出
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)

        return out

# 测试多头注意力
n_heads = 4
head_size = 8  # 4 * 8 = 32，正好回到了原始通道维度 C
mha = MultiHeadAttention(n_heads, head_size)
out = mha(x)

print(f"输入形状: {x.shape}")
print(f"多头输出形状: {out.shape}")

# 验证残差连接的可能性
print(f"它们能否相加? {'能' if x.shape == out.shape else '不能'}")

**Guided reflection**:
In `MultiHeadAttention`, we concatenate four 8-dimensional heads into a 32-dimensional vector. This has roughly the same number of parameters as using one 32-dimensional single head directly. So why does multi-head attention usually work better?


**Guided reflection**:
Why do we add `self.proj` at the end of `MultiHeadAttention`? Apart from adjusting dimensions so that residual connections fit, what role does it play in fusing information from multiple heads?


### 3.9 Deeper Observation: Differences Between Heads

To verify whether “multiple heads really look at different things,” we can randomly initialize a multi-head layer and observe the weight matrices `wei` produced by two different heads. You will see that even with exactly the same input, their attention patterns can be very different.


In [ ]:
# 观察不同头的注意力分布差异
head1 = mha.heads[0]
head2 = mha.heads[1]

with torch.no_grad():
    # 手动获取两个头的注意力矩阵
    k1, q1 = head1.key(x), head1.query(x)
    wei1 = (q1 @ k1.transpose(-2, -1)) * (head_size**-0.5)
    wei1 = F.softmax(wei1.masked_fill(tril[:T, :T] == 0, float('-inf')), dim=-1)

    k2, q2 = head2.key(x), head2.query(x)
    wei2 = (q2 @ k2.transpose(-2, -1)) * (head_size**-0.5)
    wei2 = F.softmax(wei2.masked_fill(tril[:T, :T] == 0, float('-inf')), dim=-1)

print("头 1 的最后时刻注意力分布:\n", wei1[0, -1])
print("\n头 2 的最后时刻注意力分布:\n", wei2[0, -1])
print("\n两个分布之间的差异 (欧氏距离):", torch.dist(wei1, wei2).item())

## Phase 4: Implementing the Remaining Transformer Components
### Exercise 4.1: Implementing the Feed-Forward Network

In a Transformer block, Self-Attention aggregates contextual information, while the Feed-Forward layer independently applies a nonlinear transformation to the vector at each position.


In [ ]:
class FeedForward(nn.Module):
    """ 一个简单的线性层，后跟一个非线性激活函数 """
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: 实现第一层线性变换，将维度从 n_embed 扩大到 4 * n_embed
            # TODO: 添加 ReLU 激活函数
            # TODO: 实现第二层线性变换，将维度缩回到 n_embed
            # TODO: 添加一个投影层的 Dropout (概率设为 0.2)
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(0.2),
        )

    def forward(self, x):
        return self.net(x)

# 测试 FFN
ffn = FeedForward(C)
out_ffn = ffn(out)
print(f"FFN 输入形状: {out.shape}")
print(f"FFN 输出形状: {out_ffn.shape}")

**Guided reflection**: Why do we usually expand the hidden dimension inside the FFN, for example to 4 times the size, instead of keeping it equal to `n_embed`? Intuitively, what is the purpose of this “expand first, then compress” structure?

A: This “increase dimension, activate, decrease dimension” structure essentially gives the model a larger feature space. In the high-dimensional space of $4 \times n_{embed}$, the model can learn more detailed and nonlinear feature combinations, like drawing on a larger scratchpad and then summarizing the result back into the original dimension. This is also called a Point-wise Feed-Forward network.


### Exercise 4.2: Layer Normalization

In a Transformer, we apply LayerNorm before each sublayer, namely Attention and FFN. This helps stabilize the training process.


In [ ]:
class LayerNorm(nn.Module):
    """
    LayerNorm 的简化实现。注意它与 BatchNorm 的区别：
    BatchNorm 是在 Batch 维度归一化，而 LayerNorm 是在 Channel 维度归一化。
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        # TODO: 用nn.Parameter初始化两个可学习参数：gamma (缩放) 和 beta (平移)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # TODO: 计算 x 在最后一个维度上的均值和方差
        # TODO: 执行归一化：(x - mean) / sqrt(var + eps)
        # TODO: 应用 gamma 和 beta 进行仿射变换
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True)
        eps = 1e-8
        x = (x - mean) / torch.sqrt(var + eps)
        x = x * self.gamma + self.beta

        return x

# 测试 LayerNorm
module = LayerNorm(C)
x_norm = module(x)
print(f"归一化后的均值 (应接近 0): {x_norm.mean().item():.4f}")
print(f"归一化后的标准差 (应接近 1): {x_norm.std().item():.4f}")

### Exercise 4.3: Assembling the Transformer Block

Now we reach the most exciting moment in the whole architecture: **integrating the Transformer Block**.

A standard Transformer Block is like a **LEGO brick**. It connects the components we wrote earlier in the following order:
1. **Layer Normalization (LayerNorm)**
2. **Multi-Head Attention**
3. **Residual Connection (+x)**
4. **Layer Normalization (LayerNorm)**
5. **Feed-Forward Network**
6. **Residual Connection (+x)**

**Core design**:
- **Pre-Norm structure**: in modern Transformers such as GPT-2, we usually normalize before entering each sublayer, Attention or FFN. This has been shown to make gradient flow more stable and deep networks easier to train.
- **Residual Connection**: the formula is `x = x + sublayer(x)`.

**Guided reflection**:
Why do we use residual connections? Without this simple addition, what would happen to derivative computation during backpropagation when we stack 12 layers or even 96 layers, as in GPT-3?


In [ ]:
class Block(nn.Module):
    """ Transformer Block: 通信 (Attention) + 计算 (FFN) """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding 维度, n_head: 我们想要的头数
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)

    def forward(self, x):
        # TODO: 实现 Pre-Norm 结构下的残差连接
        # 1. x = x + Attention(LayerNorm(x))
        # 2. x = x + FFN(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# 测试 Block
block = Block(n_embed=C, n_head=4)
out_block = block(x)
print(f"输入形状: {x.shape}")
print(f"Block 输出形状: {out_block.shape}")

## Phase 5: Final Integration — Building a GPT Language Model

Now we will stack the previous `Block`s and add **Positional Embedding**, because the Transformer attention mechanism itself is position-agnostic. We need to tell the model the specific position of each character in the sequence.


In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size):
        super().__init__()
        # 1. Token 嵌入表
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # 2. 位置 嵌入表 (让模型知道字符的顺序)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        # 3. 堆叠多个 Transformer Block
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        # 4. 最后的层归一化
        self.ln_f = LayerNorm(n_embed)
        # 5. 语言模型头 (从特征空间映射回词表大小)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # tok_emb: (B, T, C), pos_emb: (T, C)
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb # 融合内容信息和位置信息

        x = self.blocks(x)    # 通过所有 Block
        x = self.ln_f(x)      # 最终归一化
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 限制上下文长度不能超过 block_size
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # 只关注最后一个时刻
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# 参数定义与初始化
n_layer = 4
gpt_model = GPTLanguageModel(vocab_size, C, n_heads, n_layer, block_size)
print(f"GPT 模型参数量: {sum(p.numel() for p in gpt_model.parameters())/1e3:.1f} K")

### Deep Dive: The Physical Nature of Position Embedding

In a Transformer, the Attention mechanism is completely position-agnostic, like throwing a bag of words into a blender. To recover word order, we introduce position embeddings. Here are three core ideas.

#### 1. Why “addition” instead of “concatenation”?
Intuitively, Concatenation may seem better at preserving the independence of information, but it significantly increases dimensionality. The cleverness of **Addition** lies in using the **sparsity** of high-dimensional space:
* **Mathematical principle**: in high-dimensional space, random vectors are almost always orthogonal. The model can learn to use some dimensions for semantics and other dimensions for position. Because linear transformations are additive, $W(x + p) = Wx + Wp$, later linear layers are fully capable of disentangling the two.
* **Physical analogy**: semantics are like “color,” and position is like “brightness.” We overlay the two, and the model can still separate color from brightness through a “filter,” namely a weight matrix.

#### 2. Learning objective: from randomness to an “ordered manifold”
At initialization, each row of `position_embedding_table` is random, and the model has no sense of “position 1” versus “position 8.” During training:
* The model needs to capture neighboring words through Attention. To make the dot product $Q \times K$ higher at nearby distances, the model will push adjacent position vectors to become **closer** in vector space, or to form some specific periodic trajectory.
* Eventually, these vectors may form a **low-dimensional manifold**, similar to a spiral, allowing the model to sense “relative displacement.”

#### 3. Limitation: the “wall” of absolute positions
Because we use `nn.Embedding(block_size, n_embed)`, this is an **absolute position encoding**. The model only recognizes positions whose indices are between $0$ and `block_size-1`. If the 9th word appears at inference time beyond the learned range, the model cannot handle it because it has never seen the corresponding “position identity card.” This is why modern models such as Llama turn to Rotary Position Embedding (RoPE) for better extrapolation ability.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 获取训练前的 Position Embedding 权重
pos_weights = gpt_model.position_embedding_table.weight.detach().cpu().numpy()

plt.figure(figsize=(10, 4))
sns.heatmap(pos_weights, cmap='viridis')
plt.title("Position Embedding Matrix (Initial State)")
plt.xlabel("Embedding Dimension (C)")
plt.ylabel("Position Index (T)")
plt.show()

**Guided reflection**: As training progresses, do you think the vectors of adjacent positions such as 0 and 1 will become closer or farther apart?


## Phase 6: Training and Evaluation

We will define an `estimate_loss` function. It temporarily sets the model to evaluation mode with `eval()`, computes the average loss over multiple batches, and then switches the model back to training mode with `train()`. This is a standard industrial practice that filters out random noise during training.


In [ ]:
eval_iters = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpt_model.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    gpt_model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size)
            X, Y = X.to(device), Y.to(device)
            logits, loss = gpt_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    gpt_model.train()
    return out

# 重新定义优化器
optimizer = torch.optim.AdamW(gpt_model.parameters(), lr=1e-3, betas=(0.9, 0.9), eps=1e-7)

for iter in range(10000):
    # 1. 定期评估 Loss
    if iter % 1000 == 0 or iter == 9999:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # 2. 训练采样
    xb, yb = get_batch('train', batch_size)
    xb, yb = xb.to(device), yb.to(device)

    # 3. 梯度下降
    logits, loss = gpt_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

### Acceptance Moment: Generating Text

After the model is trained, we can start it from an empty character, ID 0, and see whether it has learned Shakespeare’s phrasing and sentence style.


In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(gpt_model.generate(context, max_new_tokens=500)[0].tolist()))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 获取训练后的 Position Embedding 权重
pos_weights = gpt_model.position_embedding_table.weight.detach().cpu().numpy()

plt.figure(figsize=(10, 4))
sns.heatmap(pos_weights, cmap='viridis')
plt.title("Position Embedding Matrix (Trained State)")
plt.xlabel("Embedding Dimension (C)")
plt.ylabel("Position Index (T)")
plt.show()

### Advanced 1: Exploring BPE Tokenization

Better Tokens: character-level tokenization is simple, but it is also the most inefficient. The standard answer used by modern large models such as GPT-4 and Llama is BPE, Byte Pair Encoding.

Why is character-level tokenization not enough?
- The sequence is too long: each letter occupies one position. Processing a book requires a length on the order of millions, while Transformer computation grows quadratically with sequence length.
- Semantics are fragmented: `'t'`, `'h'`, and `'e'` have no meaning by themselves. Only when combined as `'the'` do they carry meaning. Character-level modeling forces the model to learn these basic combinations, wasting parameters.

**What is BPE?**

BPE is a Subword algorithm. It starts from characters and repeatedly merges the most frequent adjacent pair.

For example, if `t` and `h` often appear together, they are merged into a new Token `th`.
If `th` and `e` often appear together, they are merged again into `the`.
As a result, common words such as `the` and `apple` become single Tokens, while rare words such as `unfamiliar` are split into subwords such as `un-familiar`. This perfectly balances vocabulary size and sequence length.

We will compare character-level encoding with the `cl100k_base` encoding used by OpenAI GPT-4. Observe the number of Tokens produced for the same sentence under different encodings.


In [ ]:
import tiktoken

# 准备测试文本
test_sentence = "To be, or not to be, that is the question."

# 1. 字符级编码 (我们之前的方案)
char_encoded = encode(test_sentence)

# 2. GPT-4 BPE 编码
enc = tiktoken.get_encoding("cl100k_base")
bpe_encoded = enc.encode(test_sentence)

print(f"原始文本长度: {len(test_sentence)}")
print(f"字符级 Token 数量: {len(char_encoded)}")
print(f"GPT-4 BPE Token 数量: {len(bpe_encoded)}")
print(f"\nBPE 编码内容: {bpe_encoded}")
print(f"BPE 还原结果: {[enc.decode([t]) for t in bpe_encoded]}")

### Why does this matter?

1. **Compression ratio**: You can see that BPE uses far fewer Tokens than the number of characters. This means that under the same `block_size`, a BPE-encoded model can “read” a longer context.
2. **Efficiency**: each layer of the model computes one representation per Token. Fewer Tokens means faster inference.
3. **Multilingual capability**: BPE can handle any language, including Chinese. Usually, one Chinese character corresponds to one or more subword Tokens.


### Advanced 2: Scaling and Hyperparameter Optimization

Now we will try training a slightly larger model. To handle a deeper network, we will:
1. **Increase model scale**: increase the number of layers and the embedding dimension.
2. **Manage hyperparameters**: keep the configuration centralized.
3. **Learning-rate warmup and decay (Optional)**: although we keep it simple here, this is standard practice in large-scale training.


In [ ]:
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. 切换到 BPE 编码器并重新准备数据
enc = tiktoken.get_encoding("cl100k_base")
bpe_vocab_size = enc.n_vocab
data = torch.tensor(enc.encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# 2. 超参数配置
batch_size = 32
block_size = 128
max_iters = 5000
learning_rate = 3e-4
eval_interval = 500
n_embed = 256
n_head = 4
n_layer = 6
dropout = 0.2

# 3. 重新定义所有组件以确保参数匹配
class Head(nn.Module):
    def __init__(self, n_embed, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embed):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embed)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
# 4. 初始化模型
final_model = GPTLanguageModel(bpe_vocab_size, n_embed, n_head, n_layer, block_size)
final_model.to(device)

print(f"Final BPE-GPT 参数量: {sum(p.numel() for p in final_model.parameters())/1e6:.2f} M")

# 5. 训练循环
optimizer = torch.optim.AdamW(final_model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0:
        final_model.eval()
        eval_loss = 0
        with torch.no_grad():
            for _ in range(10):
                ix = torch.randint(0, len(val_data) - block_size, (batch_size,))
                xb = torch.stack([val_data[i:i+block_size] for i in ix]).to(device)
                yb = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)
                _, loss = final_model(xb, yb)
                eval_loss += loss.item()
        print(f"Step {iter}: Val Loss {eval_loss/10:.4f}")
        final_model.train()

    ix = torch.randint(0, len(train_data) - block_size, (batch_size,))
    xb = torch.stack([train_data[i:i+block_size] for i in ix]).to(device)
    yb = torch.stack([train_data[i+1:i+block_size+1] for i in ix]).to(device)

    logits, loss = final_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 100 == 0:
        print(f"Iter {iter}: Train Loss {loss.item():.4f}")

Intermediate result check (Inference Check)

Let us see the text generated by the 27M BPE model after several training steps. Note that because the vocabulary is large and the model has more parameters, early training may focus more on learning word spelling.


In [ ]:
# 切换到评估模式
final_model.eval()

# 从空字符（或者是莎士比亚剧本中常见的开头）开始生成
# 在 BPE 中，我们直接用 enc.encode("\n") 作为起始符
context = torch.tensor([enc.encode("\n")], dtype=torch.long, device=device)

# 生成 200 个 Token (这大约相当于之前的 600-800 个字符)
generated_tokens = final_model.generate(context, max_new_tokens=200)[0].tolist()

print("--- 27M BPE-GPT 阶段性生成结果 ---")
print(enc.decode(generated_tokens))
print("----------------------------------")

# 切回训练模式继续后续训练（如果需要）
# final_model.train()

## Phase 7: Advanced Training Strategies
### 7.1 Learning Rate Scheduler

When training large-scale models, a fixed learning rate often causes two problems:
1. **Initial instability**: gradients are large at the beginning, and using a high learning rate directly may make the Loss explode.
2. **Late-stage oscillation**: near the end of training, if the step size is still large, the model may have difficulty settling into the deepest “valley.”

We will implement a typical **Warmup + Cosine Decay** scheduler:
* **Warmup**: for the first few hundred steps, linearly increase the learning rate from 0 to the target value.
* **Cosine Decay**: afterward, decrease the learning rate along a cosine curve to a very small minimum value.


In [ ]:
import math

# 学习率调度配置
warmup_iters = 200
lr_decay_iters = max_iters # 通常等于总迭代次数
min_lr = 3e-5 # 最终衰减到的最小值，通常是 learning_rate / 10

def get_lr(it):
    # 1) 线性预热阶段 (linear warmup)
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    # 2) 超过衰减阶段，返回最小学习率
    if it > lr_decay_iters:
        return min_lr
    # 3) 余弦衰减阶段
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # 从 1 降到 0
    return min_lr + coeff * (learning_rate - min_lr)

# 演示学习率曲线
lrs = [get_lr(i) for i in range(max_iters)]
plt.plot(lrs)
plt.title("Learning Rate Schedule (Warmup + Cosine Decay)")
plt.xlabel("Iterations")
plt.ylabel("Learning Rate")
plt.show()

### 7.2 Weight Decay and Gradient Clipping

To train the 27M model more robustly, we introduce the following strategies:
- **Decoupled Weight Decay**: apply decay only to weight matrices with dimension $\ge 2$.
- **Grad Clipping**: limit the total norm of the gradient to prevent gradient explosion.


In [ ]:
def get_optimizer(model, weight_decay, learning_rate, betas):
    # 提取所有需要梯度的参数
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
    
    # 区分需要 decay 和不需要 decay 的组
    # 规则：2D 权重矩阵（Linear.weight）需要 decay，1D（bias, LayerNorm.weight/beta）不需要
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas)
    return optimizer

# 配置参数
weight_decay = 0.1
grad_clip = 1.0

# 重新初始化优化器
optimizer = get_optimizer(final_model, weight_decay, learning_rate, (0.9, 0.95))
print(f"优化器已更新：启用了权重衰减 ({weight_decay})")

### 7.3 Integration: Complete Training Loop

Now we combine all techniques: **BPE Tokenization**, **27M Transformer**, **LR Scheduler**, **Weight Decay**, and **Grad Clipping**. You can run this loop to observe model convergence.


In [ ]:
# 1. 重新初始化模型
final_model = GPTLanguageModel(bpe_vocab_size, n_embed, n_head, n_layer, block_size)
final_model.to(device)

# 2. 重新配置优化器（包含权重衰减）
weight_decay = 0.1
grad_clip = 1.0
optimizer = get_optimizer(final_model, weight_decay, learning_rate, (0.9, 0.9))

print(f"开始完全体训练（参数量: {sum(p.numel() for p in final_model.parameters())/1e6:.2f} M）...")

for iter in range(max_iters):
    # 1. 动态调整学习率 (Warmup + Cosine Decay)
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # 2. 获取数据与前向传播
    ix = torch.randint(0, len(train_data) - block_size, (batch_size,))
    xb = torch.stack([train_data[i:i+block_size] for i in ix]).to(device)
    yb = torch.stack([train_data[i+1:i+block_size+1] for i in ix]).to(device)

    logits, loss = final_model(xb, yb)

    # 3. 反向传播与优化
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # 4. 梯度裁剪：防止梯度爆炸，维持训练稳定性
    torch.nn.utils.clip_grad_norm_(final_model.parameters(), grad_clip)

    optimizer.step()

    if iter % 200 == 0 or iter == max_iters - 1:
        print(f"Iter {iter}: Loss {loss.item():.4f}, Current LR: {lr:.2e}")

print("训练完成！")

### 7.4 Cross-Model Evaluation Metrics: Perplexity & BPC

Because BPE and character-level models have different vocabulary sizes, directly comparing Loss is unfair. We introduce two general metrics:
1. **Perplexity (PPL)**: $\exp(\text{loss})$. It reflects how many choices, on average, the model hesitates among when choosing the next Token.
2. **Bits Per Character (BPC)**: $\frac{\text{Total Loss} \times \log_2(e)}{\text{Total Characters}}$. It normalizes Loss to the original character dimension. **This is a comprehensive standard for evaluating “Tokenizer + model prediction ability.”** Regardless of how the vocabulary changes, the lower the BPC, the more efficiently the model compresses and predicts the underlying information.


In [ ]:
import math

@torch.no_grad()
def evaluate_metrics(model, data_source, num_batches=50):
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    for _ in range(num_batches):
        ix = torch.randint(0, len(data_source) - block_size, (batch_size,))
        xb = torch.stack([data_source[i:i+block_size] for i in ix]).to(device)
        yb = torch.stack([data_source[i+1:i+block_size+1] for i in ix]).to(device)
        
        _, loss = model(xb, yb)
        total_loss += loss.item() * (batch_size * block_size)
        total_tokens += (batch_size * block_size)

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    # 计算 BPC (假设每个 Token 对应的平均字符数可以通过数据集比例估算)
    # 莎士比亚原始字符数 / BPE Token 数
    chars_per_token = 1115394 / len(data)
    bpc = avg_loss * math.log2(math.e) / chars_per_token
    
    model.train()
    return {"Loss": avg_loss, "Perplexity": perplexity, "BPC": bpc}

# 执行评估
metrics = evaluate_metrics(final_model, val_data)
print(f"验证集评估结果:")
print(f"- Cross Entropy Loss: {metrics['Loss']:.4f}")
print(f"- Perplexity: {metrics['Perplexity']:.2f}")
print(f"- BPC (Bits Per Character): {metrics['BPC']:.4f}")

Interpreting the results:
- Perplexity ($\approx 1700$): this means that when predicting the next BPE Token, the model is on average hesitating among about 1710 possible words or subwords. For a vocabulary of around 100,000 entries, this is already a major improvement.

- Physical meaning: BPC 2.9 means that the model believes only about 2.9 bits of information are needed on average to determine one character in the Shakespeare plays. For comparison, if we guessed completely at random with probability $1/65$, BPC would be $\log_2(65) \approx 6.0$. Your model has already discovered more than half of the information structure in the data.

- Horizontal comparison: on the Shakespeare dataset, top character-level GPT models can usually reach about 1.4 to 1.6 BPC. The current 2.9 shows that the model has reached the beginner level, but because the number of training steps is still insufficient, it has not yet captured deeper literary rhetoric and logic.


---

## Phase 8: Downstream Task Transferability

The real strength of large language models (LLMs) lies in their **generalization ability**. A model pretrained on large amounts of text data can adapt well to many downstream tasks through Fine-tuning or In-context Learning without requiring major structural changes, such as:

1. **Text classification**: sentiment analysis, news classification, and so on.
2. **Question answering systems**: extracting answers from text.
3. **Text summarization**: summarizing long documents into short summaries.
4. **Machine translation**: translating one language into another.
5. **Code generation/completion**: helping programmers write code.

**Why does a pretrained model have this ability?**

By predicting the next Token, the model is forced to learn the deep structure of language, semantic relationships, factual knowledge, and a world model. The learned “Representations” are general and can serve as strong starting points for many specific tasks.

**How is transfer implemented?**

* **Fine-tuning**: add a simple task-specific layer, such as a linear classifier, on top of the pretrained model, and continue training on a small dataset for the target task.
* **In-context Learning**: without modifying model parameters, provide a task description and a few examples in the input to guide the model to complete a new task. This is the powerful ability shown by large LLMs such as GPT-3.

**Reflection**: How does a character-level or BPE-level language model “learn” the grammar and semantic rules of language during pretraining?


### 8.1: Text Classification Example

We will demonstrate how to adapt a pretrained `GPTLanguageModel` to a **text classification** task. Suppose we want to perform sentiment classification and classify text as positive or negative. The core idea is:

1. **Reuse the pretrained feature extractor**: keep all layers of `GPTLanguageModel` except the final `lm_head`, and use them to generate context-dependent text embeddings.
2. **Add a classification head**: add a simple linear layer on top of the model to map text embeddings to classification categories, for example 2 classes representing positive and negative.
3. **Freeze or fine-tune**: we can choose to freeze the pretrained layers and train only the classification head, which is effective on small datasets; or unfreeze part or all of the layers for fine-tuning.

We will build a `GPTClassifier` class that inherits most of the structure of `GPTLanguageModel` and replaces its `lm_head`.


In [ ]:
class GPTClassifier(GPTLanguageModel):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, num_classes=2):
        super().__init__(vocab_size, n_embed, n_head, n_layer, block_size)
        # 移除原有的语言模型头
        del self.lm_head

        # 添加一个新的分类头
        self.classification_head = nn.Linear(n_embed, num_classes)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # 沿用 GPTLanguageModel 的前向传播直到 ln_f
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        # 对于分类任务，我们通常只取序列中最后一个 token 的表示
        # 或者使用一个特殊的 [CLS] token 的表示 (这里我们简化为最后一个 token)
        # 假设最后一个 token 的表示聚合了整个序列的信息
        last_token_embedding = x[:, -1, :]

        # 通过分类头获得分类 logits
        logits = self.classification_head(last_token_embedding) # (B, num_classes)

        loss = None
        if targets is not None:
            # 计算分类损失 (例如交叉熵损失)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# 1. 实例化分类模型
# 我们将使用与之前 GPT 模型相同的配置，但添加 2 个输出类别
num_classes = 2 # 例如：正面/负面情感
classifier_model = GPTClassifier(bpe_vocab_size, n_embed, n_head, n_layer, block_size, num_classes)
classifier_model.to(device)

# 2. 从预训练的 GPT 模型加载权重
# 假设我们已经训练好了 final_model，将其权重加载到 classifier_model 中
# 注意：lm_head 的权重不会被加载，因为它已经被替换。
# 同时，我们需要处理 lm_head 在 state_dict 中的键，通常通过过滤或重命名来实现。
pretrained_state_dict = final_model.state_dict()

# 过滤掉 lm_head 相关的键
filtered_state_dict = {k: v for k, v in pretrained_state_dict.items() if 'lm_head' not in k}

# 加载权重，strict=False 允许 state_dict 中有未匹配的键 (即 classification_head)
classifier_model.load_state_dict(filtered_state_dict, strict=False)
print("预训练模型权重已加载到分类器模型。")

# 3. 冻结大部分层，只训练分类头
for name, param in classifier_model.named_parameters():
    if 'classification_head' not in name: # 除了分类头，其他层都冻结
        param.requires_grad = False
    else:
        param.requires_grad = True

print("冻结了除分类头以外的所有层。")
print("需要训练的参数:")
for name, param in classifier_model.named_parameters():
    if param.requires_grad:
        print(f"- {name}")

# 4. 模拟一个分类任务的前向传播
# 假设有一个 Batch 的输入文本 (B, T) 和对应的标签 (B,)
# 这里我们创建一个模拟数据
dummy_input_idx = torch.randint(0, bpe_vocab_size, (batch_size, block_size)).to(device) # 随机 token IDs
dummy_labels = torch.randint(0, num_classes, (batch_size,)).to(device) # 随机标签 (0或1)

logits, loss = classifier_model(dummy_input_idx, dummy_labels)
print(f"\n模拟分类 logits 形状: {logits.shape}") # (Batch, num_classes)
print(f"模拟分类 Loss: {loss.item():.4f}")

# 现在你可以用一个新的优化器和分类数据来训练 classifier_model 了。
# 例如:
# classification_optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=1e-4)
# for epoch in range(num_epochs):
#     # 获取分类任务的 batch (text_batch, label_batch)
#     logits, loss = classifier_model(text_batch, label_batch)
#     loss.backward()
#     classification_optimizer.step()
#     classification_optimizer.zero_grad()

#### Quick Start for Classification Fine-Tuning
Run the code block below to fine-tune the classification head for 5 quick steps on simulated data and predict the label of a test sentence:


In [ ]:
# 1. 准备实际样本数据
sample_texts = [
    "I love this Shakespeare play, it is beautiful!", # Positive
    "The tragedy was moving and well written.",     # Positive
    "This performance was terrible and boring.",    # Negative
    "I did not enjoy the show at all.",             # Negative
    "A masterpiece of English literature."          # Positive
]
# 1代表正面，0代表负面
sample_labels = torch.tensor([1, 1, 0, 0, 1], device=device)

# 2. 简单微调循环
optimizer = torch.optim.AdamW(classifier_model.classification_head.parameters(), lr=1e-3)
classifier_model.train()

print("正在使用实际文本样本微调分类头...")
for epoch in range(20):
    # 将文本转换为 Token IDs 并填充/截断到 block_size
    input_ids = []
    for text in sample_texts:
        tokens = enc.encode(text)
        # 简单的填充或截断逻辑
        if len(tokens) > block_size: tokens = tokens[:block_size]
        else: tokens += [0] * (block_size - len(tokens))
        input_ids.append(tokens)
    
    xb = torch.tensor(input_ids, device=device)
    yb = sample_labels

    logits, loss = classifier_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 3. 实时预测示例
classifier_model.eval()
with torch.no_grad():
    test_text = "This play is a complete waste of time."
    test_tokens = enc.encode(test_text)
    # 确保长度匹配
    if len(test_tokens) > block_size: test_tokens = test_tokens[:block_size]
    else: test_tokens += [0] * (block_size - len(test_tokens))
    
    test_idx = torch.tensor([test_tokens], device=device)
    logits, _ = classifier_model(test_idx)
    probs = torch.softmax(logits, dim=-1)
    prediction = torch.argmax(probs).item()

print(f"\n测试文本: {test_text}")
print(f"预测结果: {'正面' if prediction == 1 else '负面'} (置信度: {probs[0, prediction]:.2%})")

### 8.2: Building a Conversational Model

One of the most intuitive applications of large language models is as a **Conversational Agent**. This is basically a direct extension of their **text generation ability**. The core ideas are:

1. **Maintain context**: the model needs to remember the whole conversation history to generate relevant replies.
2. **Incremental generation**: after each user input, add it to the history, then let the model continue generating new content as the response.
3. **Stopping condition**: define when the model should stop generating, for example when it produces a period, a newline, or reaches the maximum length.

We can directly use the previously trained `final_model` for conversation.

**Note**: even a small model like `final_model` may not generate perfectly coherent or logical dialogue, but it will try to imitate the structure and style of language.


In [ ]:
# 切换到评估模式
final_model.eval()

# BPE 编码器
enc = tiktoken.get_encoding("cl100k_base")

def chat(model, max_new_tokens=100, stop_token='\n'):
    conversation_history = ""
    print("--- 开始对话 (输入 'exit' 结束) ---")

    while True:
        user_input = input("你: ")
        if user_input.lower() == 'exit':
            break

        # 将用户输入添加到历史中
        conversation_history += user_input + "\n"

        # 编码对话历史
        context_tokens = enc.encode(conversation_history)
        # 限制上下文长度不超过 block_size
        context_tokens = context_tokens[-block_size:]
        context = torch.tensor([context_tokens], dtype=torch.long, device=device)

        # 生成回复
        generated_tokens_tensor = model.generate(context, max_new_tokens=max_new_tokens)
        generated_tokens = generated_tokens_tensor[0].tolist()

        # 解码所有生成的 Token，然后提取新增的部分
        full_generated_text = enc.decode(generated_tokens)
        # 找出新生成的文本，从历史长度开始
        model_response = full_generated_text[len(conversation_history):]

        # 寻找停止 Token 并截断
        if stop_token in model_response:
            model_response = model_response.split(stop_token, 1)[0] + stop_token
        
        # 打印模型回复
        print(f"模型: {model_response.strip()}")
        
        # 更新对话历史，包含模型回复
        conversation_history += model_response

    print("--- 对话结束 ---")

# 运行对话函数
# 可以调整 max_new_tokens 来控制模型回复的长度
chat(final_model, max_new_tokens=80, stop_token='\n')

#### 8.2.1: Example of Conversational Fine-Tuning

We will simulate a small dialogue dataset and show how to use it to fine-tune `final_model`. The usual data format is `user_turn` + `model_response`, so that the model learns to predict the next reply.


In [ ]:
import random

# 模拟一个小型对话数据集
# 每个元组包含 (对话历史, 预期模型回复)
# 为了简化，这里每个条目都假设是独立的对话片段，
# 实际中，需要构建完整的对话链
dialogue_dataset = [
    ("Hello, how are you?", "I'm doing well, thank you! How can I assist you today?"),
    ("Tell me about Shakespeare.", "William Shakespeare was an English playwright, poet, and actor, widely regarded as the greatest writer in the English language."),
    ("What is the meaning of life?", "That is a profound philosophical question that has been debated for centuries. There is no single universally accepted answer."),
    ("Can you write a poem?", "Certainly! What topic would you like the poem to be about?"),
    ("I need help with my code.", "I can help with coding. What kind of problem are you facing?"),
    ("Who is your creator?", "I am a large language model, trained by Google."),
    ("How old are you?", "I do not have an age in the human sense, as I am an AI."),
    ("Goodbye.", "Goodbye! Have a great day."),
    ("What's your favorite book?", "As an AI, I don't read books or have preferences like humans do, but I can process information from countless texts!"),
    ("Thank you!", "You're most welcome! Is there anything else I can help with?"),
    ("Where are you from?", "I exist as a computer program, so I don't have a physical origin in the way humans do."),
    ("What do you think of this play?", "I can analyze the text of the play if you provide it. What aspects are you interested in?"),
    ("I am feeling sad.", "I'm sorry to hear that. Is there anything specific you'd like to talk about or any information I can provide?"),
    ("Happy to see you.", "It's good to 'see' you too! How may I assist you?")
]

# 将对话数据转换为 token ID 序列
def prepare_dialogue_batch(batch_size):
    xb_dialogue, yb_dialogue = [], []
    for _ in range(batch_size):
        # 随机选择一个对话片段
        user_turn, model_response = random.choice(dialogue_dataset)
        # 拼接用户输入和模型响应，用换行符分隔作为训练目标
        full_text = user_turn + "\n" + model_response + "\n"
        
        # 编码整个序列
        encoded_full_text = enc.encode(full_text)
        
        # 确保序列长度不超过 block_size
        if len(encoded_full_text) < block_size + 1:
            # x 是除了最后一个 token 的所有 token
            # y 是除了第一个 token 的所有 token (即 x 的下一个 token)
            x = encoded_full_text[:-1]
            y = encoded_full_text[1:]
            
            # 填充到 block_size
            padding_len = block_size - len(x)
            x = x + [0] * padding_len # 使用 0 (通常是 PAD token) 进行填充
            y = y + [0] * padding_len
            
            xb_dialogue.append(x)
            yb_dialogue.append(y)
        else:
            # 如果太长，就截断
            x = encoded_full_text[:block_size]
            y = encoded_full_text[1:block_size+1]
            xb_dialogue.append(x)
            yb_dialogue.append(y)

    return torch.tensor(xb_dialogue, dtype=torch.long, device=device), torch.tensor(yb_dialogue, dtype=torch.long, device=device)

# 测试数据准备
sample_xb, sample_yb = prepare_dialogue_batch(2)
print("Sample XB shape:", sample_xb.shape)
print("Sample YB shape:", sample_yb.shape)
print("Sample XB (decoded):", [enc.decode(x.tolist()) for x in sample_xb])
print("Sample YB (decoded):", [enc.decode(y.tolist()) for y in sample_yb])


#### Fine-Tuning Training Loop

We will use a training loop similar to before, but now with our simulated dialogue dataset. To avoid overwriting the knowledge learned from the Shakespeare data too much, and to focus more on dialogue patterns, we may use a smaller learning rate or train only some layers, such as unfreezing only layers near `lm_head`, or unfreezing all layers for full fine-tuning depending on the data size.

Here, we will not freeze any layers. We let the model perform end-to-end fine-tuning on the dialogue data. We use a separate optimizer and a separate number of training iterations to distinguish this fine-tuning process.


In [ ]:
# 重新初始化一个优化器，用于对话微调
# 通常使用较小的学习率进行微调
conversation_learning_rate = 1e-5
conversation_optimizer = torch.optim.AdamW(final_model.parameters(), lr=conversation_learning_rate, betas=(0.9, 0.95))

# 设置微调迭代次数
micro_finetune_iters = 1000
grad_clip = 1.0

print("\n--- 开始对话模型微调 ---")
final_model.train()

for iter_ft in range(micro_finetune_iters):
    # 获取对话 Batch 数据
    xb_ft, yb_ft = prepare_dialogue_batch(batch_size)

    # 前向传播计算 Loss
    logits_ft, loss_ft = final_model(xb_ft, yb_ft)

    # 反向传播更新参数
    conversation_optimizer.zero_grad(set_to_none=True)
    loss_ft.backward()    
    torch.nn.utils.clip_grad_norm_(final_model.parameters(), grad_clip) # 梯度裁剪
    conversation_optimizer.step()

    if (iter_ft + 1) % 100 == 0:
        print(f"Finetune Iter {iter_ft+1}: Loss {loss_ft.item():.4f}")

print("对话模型微调完成！")


#### Using the Fine-Tuned Model for Conversation

Now we run the `chat` function again and see whether the fine-tuned `final_model` improves in generating dialogue replies. It should be more likely to generate response patterns similar to those in the simulated dataset.


In [ ]:
# 切换到评估模式
final_model.eval()

# 运行对话函数
# 可以调整 max_new_tokens 来控制模型回复的长度
chat(final_model, max_new_tokens=80, stop_token='\n')

# 切回训练模式（如果需要继续训练）
# final_model.train()